In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import QuantileTransformer, MaxAbsScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor

In [2]:
# Loading the training dataset
df = pd.read_csv('cold-start-consumption-forecasting-training-data.csv', sep=';')
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df.sort_values(by='timestamp', inplace=True)

In [3]:
df[df['site_id']==2]

,obs_id,site_id,timestamp,temperature,temperature_-1,temperature_-2,load,load_-1,load_-2,target
1221786,18469,2,2013-01-01 03:00:00+00:00,0.500000,1.000000,1.000000,52561.679251,51033.723459,49964.154405,53631.248306
29192,18470,2,2013-01-01 04:00:00+00:00,0.500000,0.500000,1.000000,55159.204098,52561.679251,51033.723459,62340.596321
56052,18471,2,2013-01-01 05:00:00+00:00,0.000000,0.500000,0.500000,58979.093579,55159.204098,52561.679251,106345.723136
1332108,18472,2,2013-01-01 06:00:00+00:00,0.000000,0.000000,0.500000,58520.706841,58979.093579,55159.204098,132473.767183
1063812,18473,2,2013-01-01 07:00:00+00:00,0.000000,0.000000,0.000000,59590.275895,58520.706841,58979.093579,160129.767021
...,...,...,...,...,...,...,...,...,...,...
438680,32888,2,2014-08-24 22:00:00+00:00,18.000000,18.500000,20.166667,65396.507906,66771.668119,65854.894643,173117.391255
438681,32889,2,2014-08-24 23:00:00+00:00,17.933333,18.000000,18.500000,66313.281381,65396.507906,66771.668119,177854.054211
359458,32890,2,2014-08-25 00:00:00+00:00,18.000000,17.933333,18.000000,72730.695708,66313.281381,65396.507906,172200.617780
1054107,32891,2,2014-08-25 01:00:00+00:00,17.000000,18.000000,17.933333,77314.563085,72730.695708,66313.281381,170978.253146


In [9]:
df.site_id.unique()

array([ 2, 44, 12,  8,  7, 10, 17, 11, 29, 31, 32, 30, 33, 35,  5, 59, 72,
        9, 43, 57, 20, 42, 74, 55, 49, 75, 13, 54, 39, 76,  4,  1, 48,  3,
       73, 60, 77, 56, 50, 14, 28, 61, 68, 63, 18, 19, 16, 15,  6, 70, 65,
       64, 69, 52, 51, 71, 58, 34, 62, 21, 66, 22, 23, 24, 26, 27, 36, 37,
       38, 67, 40, 41, 53, 45, 46, 47, 25])

In [10]:
df.groupby(["site_id"]).mean().timestamp

site_id
1    2016-03-25 08:00:00+00:00
2    2013-10-28 14:30:00+00:00
3    2016-05-02 18:30:00+00:00
4    2016-03-07 11:00:00+00:00
5    2015-09-17 14:30:00+00:00
                ...           
73   2016-05-15 16:00:00+00:00
74   2015-12-23 16:00:00+00:00
75   2016-01-29 14:00:00+00:00
76   2016-02-10 06:00:00+00:00
77   2016-03-28 20:30:00+00:00
Name: timestamp, Length: 77, dtype: datetime64[ns, UTC]

In [4]:
# Preprocessing the data
df.fillna(df.mean(), inplace=True)  
X = df[['temperature', 'temperature_-1', 'temperature_-2', 'load', 'load_-1', 'load_-2']]
y = df['target']

# Data transformation
scaler = MaxAbsScaler()
X_scaled = scaler.fit_transform(X)

In [5]:
X_scaled

array([[ 0.01190476,  0.02380952,  0.02380952,  0.03185315,  0.03092719,
         0.03027901],
       [-0.03095238, -0.02428571, -0.01285714,  0.00173297,  0.00203432,
         0.00184891],
       [ 0.01190476,  0.01190476,  0.02380952,  0.03342729,  0.03185315,
         0.03092719],
       ...,
       [ 0.28392857,  0.2734127 ,  0.26547619,  0.01048122,  0.0107357 ,
         0.01007778],
       [ 0.29880952,  0.28392857,  0.2734127 ,  0.01086604,  0.01048122,
         0.0107357 ],
       [ 0.2984127 ,  0.29880952,  0.28392857,  0.01092915,  0.01086604,
         0.01048122]])

In [6]:
# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

### Issues

*   **TRAIN/TEST SPLIT IS WRONG!!!!** BECAUSE IT IS RANDOM, WHILE IT SHOULDN'T! It doesn't make sense to predict the energy in a date with another that is in the future... The split should rather be done by separating the data for each building, and then adding the 75% (left) quantile for each building in the train dataset, and then the other 25% (the most recent dates) in the validation dataset.
*   **Data leakage**: don't fill NAs (`.fillna`) with averages! That's data leakage. You know how to solve that (check ds701 labs).
*   **Train/test split occurs BEFORE it's scaled** (with MinMaxScaler for example).

### Recommendations
*   To run the AutoTS, don't use data from only 1 building. Use a few data but from several buildings: f.e. 40 hours of 50 buildings?

## Extra Tree Regressor

Model fitting:

In [7]:
models = [ExtraTreesRegressor(n_estimators=50, min_samples_leaf=1, max_depth=None) for _ in range(5)]
for model in models:
    model.fit(X_train, y_train)

# Ensemble prediction: Average predictions from all models
et_predictions = np.mean([model.predict(X_test) for model in models], axis=0)

Evaluation:

In [8]:
# Mean Squared Error
mse = mean_squared_error(y_test, et_predictions)
print(f"Mean Squared Error: {mse}")

# Adjusted R^2
r2 = r2_score(y_test, et_predictions)
n = len(et_predictions)  # Number of observations
p = 6  # Number of predictors
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("Adjusted R-squared Score:", adjusted_r2)

Mean Squared Error: 4191987539.4908276
Adjusted R-squared Score: 0.8329564107384038


**Evaluation (only with training dataset)--> MSE, Adjusted R-squared**

1) 500000 observations, #n_estimators=10 --> 3174149111.9315395, 0.8146241929823222
2) All observations, #n_estimators=50 --> 4196148449.255189, **0.8327906055457392**

In [9]:
test = pd.read_csv("cold-start-consumption-forecasting-test-data.csv", sep=";", parse_dates=['timestamp'])
test['timestamp'] = pd.to_datetime(test['timestamp'], utc=True)
test.sort_values(by='timestamp', inplace=True)
test_label = test['target']
test_data = test.drop(columns=['target'])

In [10]:
test.head()

,obs_id,site_id,timestamp,temperature,temperature_-1,temperature_-2,load,load_-1,load_-2,target
157,2280,83,2014-01-02 04:00:00+00:00,3.033333,3.000000,3.000000,1231.920396,1240.484790,1239.132517,0.0
148,2182,83,2014-01-03 15:00:00+00:00,5.000000,4.000000,4.166667,25851.396508,25731.044241,25937.040443,0.0
4329,2017,83,2014-01-06 02:00:00+00:00,1.500000,3.533333,3.500000,1259.416607,1257.613577,1252.204486,0.0
3802,2082,83,2014-01-13 15:00:00+00:00,6.000000,6.500000,6.066667,25975.354836,25906.839687,26195.324522,0.0
2819,2062,83,2014-01-14 22:00:00+00:00,5.100000,5.500000,6.000000,1301.787817,6994.404973,31049.081869,0.0


In [15]:
prueba = test[test['target']!=0]
prueba_label = prueba['target']
prueba_data = prueba[['temperature', 'temperature_-1', 'temperature_-2', 'load', 'load_-1', 'load_-2']]
scaler = MaxAbsScaler()
X_scaled3 = scaler.fit_transform(prueba_data)
et_predictions3 = np.mean([model.predict(X_scaled3) for model in models], axis=0)

ValueError: Found input variables with inconsistent numbers of samples: [1402448, 672]

In [17]:
# Mean Squared Error
mse = mean_squared_error(prueba_label, et_predictions3)
print(f"Mean Squared Error: {mse}")

# Adjusted R^2
r2 = r2_score(prueba_label, et_predictions3)
n = len(et_predictions3)  # Number of observations
p = 6  # Number of predictors
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("Adjusted R-squared Score:", adjusted_r2)

Mean Squared Error: 8153195585.023301
Adjusted R-squared Score: 0.7704605572804575


In [ ]:
test[['temperature', 'temperature_-1', 'temperature_-2', 'load', 'load_-1', 'load_-2']]

In [14]:
prueba_data

,obs_id,site_id,timestamp,temperature,temperature_-1,temperature_-2,load,load_-1,load_-2
2115,1941,83,2014-02-11 10:00:00+00:00,4.400000,3.500000,3.000000,2.463255e+04,2.500127e+04,1.508235e+04
130,1925,83,2014-02-21 14:00:00+00:00,12.500000,12.333333,12.000000,2.829450e+04,2.665284e+04,2.472225e+04
478,1935,83,2014-02-28 09:00:00+00:00,7.500000,6.000000,5.800000,2.496746e+04,2.064425e+04,7.572727e+01
4121,1922,83,2014-03-12 19:00:00+00:00,12.733333,14.000000,17.000000,3.046941e+04,3.001414e+04,2.862987e+04
2669,1920,83,2014-03-25 17:00:00+00:00,9.000000,10.200000,11.000000,2.801188e+04,2.675111e+04,2.671910e+04
...,...,...,...,...,...,...,...,...,...
215,3089,86,2017-10-30 18:00:00+00:00,18.000000,18.000000,19.000000,1.453578e+04,1.469955e+04,1.314470e+04
1382,8,78,2017-10-31 13:00:00+00:00,15.933333,15.000000,14.500000,1.027801e+06,1.006864e+06,1.044275e+06
2651,1558,82,2017-11-03 08:00:00+00:00,7.781818,7.090909,6.809091,3.249508e+04,3.104441e+04,2.146997e+04
2675,46,78,2017-11-15 08:00:00+00:00,3.133333,2.125000,1.800000,9.998315e+05,9.568210e+05,8.007903e+05


In [12]:
# Preprocessing the data
test_data.fillna(test_data.mean(), inplace=True)  
X = test[['temperature', 'temperature_-1', 'temperature_-2', 'load', 'load_-1', 'load_-2']]
y = test['target']

# Data transformation
scaler = MaxAbsScaler()
X_scaled2 = scaler.fit_transform(X)

In [13]:
et_predictions2 = np.mean([model.predict(X_scaled2) for model in models], axis=0)

In [14]:
# Mean Squared Error
mse = mean_squared_error(y, et_predictions2)
print(f"Mean Squared Error: {mse}")

# Adjusted R^2
r2 = r2_score(y, et_predictions2)
n = len(et_predictions2)  # Number of observations
p = 6  # Number of predictors
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("Adjusted R-squared Score:", adjusted_r2)

Mean Squared Error: 49969655032.35391
Adjusted R-squared Score: -8.59602072065979


In [18]:
pd.DataFrame(et_predictions2).to_csv("TestPredictions.csv")

In [19]:
final_pred = pd.DataFrame(et_predictions2)
final_pred

,0
0,2.404157e+03
1,3.340181e+04
2,2.305576e+03
3,3.429487e+04
4,2.342193e+04
...,...
5371,4.206108e+05
5372,1.074200e+06
5373,4.754389e+05
5374,1.944329e+05


In [39]:
finaldf = test[['obs_id','site_id', 'timestamp']]
finaldf["target"] = final_pred
finaldf.to_csv("FinalResults.csv")

/var/folders/rv/76gg687x2fvc_8l36z0y2xdh0000gn/T/ipykernel_6031/2568385706.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  finaldf["target"] = final_pred


In [40]:
finaldf

,obs_id,site_id,timestamp,target
157,2280,83,2014-01-02 04:00:00+00:00,975.080060
148,2182,83,2014-01-03 15:00:00+00:00,28087.119115
4329,2017,83,2014-01-06 02:00:00+00:00,412431.644203
3802,2082,83,2014-01-13 15:00:00+00:00,22095.545591
2819,2062,83,2014-01-14 22:00:00+00:00,33174.053741
...,...,...,...,...
1591,274,78,2017-11-15 03:00:00+00:00,41903.755934
2675,46,78,2017-11-15 08:00:00+00:00,46993.230074
4581,299,78,2017-11-18 14:00:00+00:00,32781.864216
2561,47,78,2017-11-19 10:00:00+00:00,16942.949504


In [53]:
merged_df = finaldf.merge(submission, on=['site_id', 'timestamp'], how='inner')
terrier = merged_df.rename(columns={'obs_id_y':'obs_id', 'target_x':'target'})
terrier = terrier[['obs_id', 'site_id', 'timestamp', 'leaderboard_target', 'target']]
terrier.to_csv("TerrierThreat.csv")

In [20]:
submission = pd.read_csv("cold-start-consumption-forecasting-submission-data.csv", sep=";", parse_dates=['timestamp'])
submission['timestamp'] = pd.to_datetime(submission['timestamp'], utc=True)
submission.sort_values(by='timestamp', inplace=True)
submission

,obs_id,site_id,timestamp,leaderboard_target,target
2880,1992,83,2014-01-02 04:00:00+00:00,private,0.0
3325,1894,83,2014-01-03 15:00:00+00:00,private,0.0
3704,1729,83,2014-01-06 02:00:00+00:00,public,0.0
1900,1794,83,2014-01-13 15:00:00+00:00,public,0.0
1602,1774,83,2014-01-14 22:00:00+00:00,public,0.0
...,...,...,...,...,...
708,242,78,2017-11-12 04:00:00+00:00,private,0.0
3803,224,78,2017-11-12 22:00:00+00:00,private,0.0
3238,226,78,2017-11-15 03:00:00+00:00,private,0.0
4413,251,78,2017-11-18 14:00:00+00:00,private,0.0


## Gradient Boosting Regressor

In [53]:
gb_model = GradientBoostingRegressor(n_estimators=75, learning_rate=0.1, max_depth=7, random_state=42)
gb_model.fit(X_train, y_train)
gb_predictions = gb_model.predict(X_test)

Evaluation:

In [54]:
#Mean Squared Error
gb_mse = mean_squared_error(y_test, gb_predictions)
print(f"Gradient Boosting MSE: {gb_mse}")

# Adjusted R^2
r2 = r2_score(y_test, gb_predictions)
n = len(predictions)  # Number of observations
p = 6  # Number of predictors in your model
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("Adjusted R-squared Score:", adjusted_r2)

Gradient Boosting MSE: 4217258399.0359507
Adjusted R-squared Score: 0.8319494098724958


*   GBM with n_estimators=100, learning_rate=0.1, max_depth=3 gives results --> Adjusted R-squared Score: 0.8269746025391094.
*   GBM with n_estimators=10, learning_rate=0.1, max_depth=7 gives results --> Adjusted R-squared Score: 0.7275775339726147.
*   GBM with n_estimators=50, learning_rate=0.1, max_depth=7 gives results --> Adjusted R-squared Score: 0.8317213619376832.
*   GBM with n_estimators=10, learning_rate=0.5, max_depth=7 gives results --> Adjusted R-squared Score: 0.8282184725816316.
*   GBM with n_estimators=50, learning_rate=0.5, max_depth=7 gives results --> Adjusted R-squared Score: 0.8203953425163166.
*   GBM with n_estimators=50, learning_rate=0.3, max_depth=7 gives results --> Adjusted R-squared Score: 0.8290866549488974.
*   GBM with n_estimators=75, learning_rate=0.3, max_depth=7 gives results --> Adjusted R-squared Score: **0.8319494098724958**.

## Tree hyperparameter tuning

In [38]:
tree = DecisionTreeRegressor()

In [39]:
# Selecting the optimal max depth
depths = []
mse_scores = []

for d in range(1, 25):
    tree = DecisionTreeRegressor(max_depth=d)
    fit = tree.fit(X_train, y_train)
    y_pred = tree.predict(X_test)
    score = mean_squared_error(y_test, y_pred)
    print(d, score)
    depths.append(d)
    mse_scores.append(score)

1 12013918749.543364
2 6269571987.115477
3 4952851806.627514
4 4590391105.243647
5 4459378943.983615
6 4388517149.205959
7 4371129256.11419
8 4378606804.50216
9 4417472388.8029175
10 4506595940.038158
11 4567671645.9466915
12 4765232449.867905
13 4945526236.470468
14 5160834000.306619
15 5266519995.165941
16 5491324769.072028
17 5700599860.4678135
18 5956666222.723023
19 6175449356.866365
20 6412178643.156171
21 6669996141.868778
22 6806056045.389877
23 7042773982.41419
24 7207874210.698074


In [40]:
min(mse_scores)

4371129256.11419

The optimal depth is d=7.